# 10 — Game-Theoretic Analysis with the 07 Rich-Pool Model

GEB makalesi için oyun-teorik analizler — Shapley değerleri ve blocking coalition analizi — `07_full_gnn_rich` modelinin öğrendiği v(S) fonksiyonu üzerinden.

## Mod seçimi

Notebook iki modda çalışır:
- **`MODE = 'load'`** (varsayılan): Drive'dan eğitilmiş `full_rich_gnn.pt` checkpoint'ini yükler. Hızlı, 5 saniye.
- **`MODE = 'train'`**: Modeli baştan eğitir (07'deki tam protokol). ~5-10 dakika A100'de.

Checkpoint dosyası bulunamazsa otomatik olarak `'train'` moduna geçer.

## Analiz planı (öncelik #1, manifesto GEB plan)

1. **Monte Carlo Shapley values** — Üye marjinal katkıları (raw logit üzerinden, sigmoid saturation'ı önlemek için)
2. **Blocking coalition analysis** — Hangi alt-koalisyonlar v(S') > v(S) sağlıyor? (non-superadditivity göstergesi)
3. **Coalition'lar:** Warsaw Pact, NATO-26, EU-27, ASEAN-10, BRICS

Diğer planlanan analizler (NBS, nucleolus, sequential formation, counterfactual) sonraki notebook'larda.

In [ ]:
# ============================================================
# YAPILANDIRMA — Mod seçimi
# ============================================================
MODE = 'load'  # 'load' = drive'dan yükle (default), 'train' = baştan eğit

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
MODELS_DIR = os.path.join(PROJECT_DIR, 'models')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Mode: {MODE}')

In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import RGCNConv
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from tqdm.auto import tqdm
import random
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = 4

In [ ]:
# Snapshot'ları yükle (heterojen, 07 ile aynı)
def load_snapshot(year):
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    data = torch.load(path, weights_only=False)
    x = data['country'].x
    cow_codes = data['country'].cow_code.numpy()
    ei_list, et_list = [], []
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index
            ei_list.append(ei)
            et_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
    if not ei_list:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_type = torch.zeros(0, dtype=torch.long)
    else:
        edge_index = torch.cat(ei_list, dim=1)
        edge_type = torch.cat(et_list)
    return {
        'x': x.to(DEVICE),
        'edge_index': edge_index.to(DEVICE),
        'edge_type': edge_type.to(DEVICE),
        'cow_codes': cow_codes,
        'code_to_idx': {int(c): i for i, c in enumerate(cow_codes)},
        'year': year,
    }

snapshots = {}
for year in range(1946, 2017):
    s = load_snapshot(year)
    if s is not None:
        snapshots[year] = s

NUM_FEATURES = next(iter(snapshots.values()))['x'].shape[1]
print(f'Snapshot: {len(snapshots)} yıl, {NUM_FEATURES} özellik')

## Model mimarisi (07 ile aynı)

In [ ]:
class RGCNEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_relations, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    def forward(self, x, ei, et):
        h = self.conv1(x, ei, et); h = self.norm1(h); h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.conv2(h, ei, et); h = self.norm2(h)
        return h

class RichCoalitionHead(nn.Module):
    def __init__(self, embed_dim, raw_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        pool_dim = 4 * (embed_dim + raw_dim) + 2
        self.W_pair = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)
        self.validity_mlp = nn.Sequential(
            nn.Linear(pool_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1))
        embed_pool_dim = 4 * embed_dim + 1
        self.duration_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Softplus())
        self.cohesion_head = nn.Sequential(
            nn.Linear(embed_pool_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, 1), nn.Sigmoid())
    def rich_pool(self, f):
        std = f.std(dim=0) if f.shape[0] > 1 else torch.zeros_like(f[0])
        return torch.cat([f.mean(dim=0), std, f.max(dim=0).values, f.min(dim=0).values])
    def forward(self, z, x_raw):
        k = z.shape[0]
        size_feat = torch.tensor([k/20.0], device=z.device)
        z_pool = self.rich_pool(z); raw_pool = self.rich_pool(x_raw)
        if k >= 2:
            Wz = z @ self.W_pair; pm = z @ Wz.T
            pair = (pm.sum() - pm.diag().sum()) / (k*(k-1))
        else:
            pair = torch.tensor(0.0, device=z.device)
        v = self.validity_mlp(torch.cat([z_pool, raw_pool, size_feat, pair.unsqueeze(0)])).squeeze()
        mt = torch.cat([z_pool, size_feat])
        return v, self.duration_head(mt).squeeze(), self.cohesion_head(mt).squeeze()

class FullRichGNN(nn.Module):
    def __init__(self, num_features, hidden_dim=128, embed_dim=128, num_relations=4, head_hidden=128, dropout=0.3):
        super().__init__()
        self.encoder = RGCNEncoder(num_features, hidden_dim, embed_dim, num_relations, dropout)
        self.head = RichCoalitionHead(embed_dim, num_features, head_hidden, dropout)
    def forward_sample(self, snap, member_idx_list):
        z_all = self.encoder(snap['x'], snap['edge_index'], snap['edge_type'])
        idx = torch.tensor(member_idx_list, dtype=torch.long, device=DEVICE)
        return self.head(z_all[idx], snap['x'][idx])

print('Model sınıfları tanımlandı')

## Model: Yükle veya Eğit

In [ ]:
ckpt_path = os.path.join(MODELS_DIR, 'full_rich_gnn.pt')

# Checkpoint yoksa otomatik 'train' moduna geç
if MODE == 'load' and not os.path.exists(ckpt_path):
    print(f'⚠ Checkpoint bulunamadı: {ckpt_path}')
    print('  → MODE otomatik olarak "train"e geçiyor')
    MODE = 'train'

model = FullRichGNN(NUM_FEATURES).to(DEVICE)

if MODE == 'load':
    state = torch.load(ckpt_path, weights_only=False)
    model.load_state_dict(state)
    model.eval()
    print(f'✓ Model yüklendi: {ckpt_path}')
    print(f'  Parametre sayısı: {sum(p.numel() for p in model.parameters()):,}')
else:
    print('Eğitim moduna girildi — aşağıdaki eğitim hücresini çalıştır')

In [ ]:
# === EĞİTİM HÜCRESİ — sadece MODE == 'train' ise çalıştır ===
if MODE == 'train':
    # Koalisyon örneklerini yükle
    pos_df = pd.read_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
    neg_df = pd.read_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))
    
    def parse_members(s):
        if isinstance(s, list): return [int(x) for x in s]
        if isinstance(s, str): return [int(x) for x in s.split(',')]
        return []
    
    def build_samples(df, label):
        out = []
        for _, row in df.iterrows():
            year = int(row['year'])
            if year not in snapshots: continue
            members = parse_members(row['members_str'])
            c2i = snapshots[year]['code_to_idx']
            valid = [m for m in members if m in c2i]
            if len(valid) < 2: continue
            out.append({
                'year': year, 'members': valid,
                'member_idx': [c2i[m] for m in valid],
                'label': label,
                'duration': float(row.get('total_duration', 0)) if label == 1 else 0.0,
                'cohesion': float(row.get('internal_vote_agreement', 0) or 0) if label == 1 else 0.0,
            })
        return out
    
    all_samples = build_samples(pos_df, 1) + build_samples(neg_df, 0)
    VAL_END = 2009
    test_samples = [s for s in all_samples if s['year'] > VAL_END]
    pre_test = [s for s in all_samples if s['year'] <= VAL_END]
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(pre_test)); rng.shuffle(idx)
    val_size = int(0.15 * len(pre_test))
    val_set = set(idx[:val_size].tolist())
    train_samples = [pre_test[i] for i in range(len(pre_test)) if i not in val_set]
    val_samples = [pre_test[i] for i in range(len(pre_test)) if i in val_set]
    
    # Pretrained encoder yükle (varsa)
    pretrain_path = os.path.join(MODELS_DIR, 'encoder_pretrained.pt')
    if os.path.exists(pretrain_path):
        ckpt = torch.load(pretrain_path, weights_only=False)
        if 'encoder_state_dict' in ckpt:
            model.encoder.load_state_dict(ckpt['encoder_state_dict'])
        else:
            model.encoder.load_state_dict(ckpt)
        print('✓ Pretrained encoder yüklendi')
    
    EPOCHS = 150; LR = 1e-3; ENCODER_LR = 1e-4; WEIGHT_DECAY = 1e-3
    LABEL_SMOOTHING = 0.1; PATIENCE = 25; POS_WEIGHT = 2.5; BATCH_SIZE = 32
    LAMBDA_DUR = LAMBDA_COH = 0.3
    
    optimizer = torch.optim.Adam([
        {'params': model.encoder.parameters(), 'lr': ENCODER_LR},
        {'params': model.head.parameters(), 'lr': LR},
    ], weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    def bce_smooth(logit, target, smoothing=0.1):
        target_s = target * (1 - smoothing) + 0.5 * smoothing
        w = torch.tensor(POS_WEIGHT if target.item() > 0.5 else 1.0, device=DEVICE)
        return F.binary_cross_entropy_with_logits(logit, target_s, weight=w)
    
    def evaluate(model, samples):
        model.eval()
        logits, labels = [], []
        with torch.no_grad():
            for s in samples:
                v, _, _ = model.forward_sample(snapshots[s['year']], s['member_idx'])
                logits.append(v.item()); labels.append(s['label'])
        logits, labels = np.array(logits), np.array(labels)
        probs = 1/(1+np.exp(-logits))
        p, r, _ = precision_recall_curve(labels, probs)
        f1 = (2*p*r/(p+r+1e-10)).max()
        return {'AUC': roc_auc_score(labels, probs),
                'PR-AUC': average_precision_score(labels, probs), 'F1_opt': f1}
    
    best_val_auc, best_state, pat = 0, None, 0
    for epoch in range(EPOCHS):
        model.train()
        random.shuffle(train_samples)
        for i in range(0, len(train_samples), BATCH_SIZE):
            batch = train_samples[i:i+BATCH_SIZE]
            optimizer.zero_grad()
            bl, nv = 0.0, 0
            for s in batch:
                v, dur, coh = model.forward_sample(snapshots[s['year']], s['member_idx'])
                target = torch.tensor(float(s['label']), device=DEVICE)
                loss = bce_smooth(v, target, LABEL_SMOOTHING)
                if s['label'] == 1:
                    dur_t = torch.log1p(torch.tensor(s['duration'], device=DEVICE))
                    loss = loss + LAMBDA_DUR * F.mse_loss(dur, dur_t)
                    if s['cohesion'] > 0:
                        coh_t = torch.tensor(s['cohesion'], device=DEVICE)
                        loss = loss + LAMBDA_COH * F.mse_loss(coh, coh_t)
                bl = bl + loss; nv += 1
            if nv == 0: continue
            (bl / nv).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        vm = evaluate(model, val_samples)
        if vm['AUC'] > best_val_auc:
            best_val_auc = vm['AUC']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            pat = 0; flag = ' ←'
        else:
            pat += 1; flag = ''
        if epoch % 5 == 0 or flag:
            print(f'Epoch {epoch:3d} val AUC={vm["AUC"]:.4f}{flag}')
        if pat >= PATIENCE: break
    
    model.load_state_dict(best_state)
    torch.save(best_state, ckpt_path)
    model.eval()
    print(f'✓ Eğitim tamamlandı, kaydedildi: {ckpt_path}')
else:
    print('(Eğitim modu değil — atlandı)')

## v(S) hesaplama yardımcı fonksiyonları

In [ ]:
def compute_v(members_cow, year, return_logit=False):
    """Verilen üye COW kodları + yıl için v(S) skoru.
    
    Args:
        members_cow: COW kodu listesi
        year: yıl
        return_logit: True ise raw logit, False ise sigmoid(logit) döndürür
    """
    if year not in snapshots:
        return None
    snap = snapshots[year]
    valid = [m for m in members_cow if m in snap['code_to_idx']]
    if len(valid) < 2:
        return None
    idx = [snap['code_to_idx'][m] for m in valid]
    with torch.no_grad():
        logit, _, _ = model.forward_sample(snap, idx)
    if return_logit:
        return logit.item()
    return torch.sigmoid(logit).item()

# COW code → country name mapping
country_master = pd.read_parquet(os.path.join(CANON_DIR, 'country_master.parquet'))
cow_to_name = {}
for _, row in country_master.iterrows():
    cow_to_name[int(row['cow_code'])] = row.get('state_name', f"COW{row['cow_code']}")

def names(cow_list):
    return [cow_to_name.get(c, str(c)) for c in cow_list]

# Sanity check — birkaç koalisyonu skorla
test_coalitions = {
    'NATO-26 (2010)': ([2,20,200,210,211,212,220,230,235,255,290,310,316,317,325,349,350,355,360,366,367,368,385,390,395,640], 2010),
    'Varşova Paktı (1980)': ([265,290,310,315,355,360,365], 1980),
    'BRICS (2014)': ([140,365,560,710,750], 2014),
}

print('Sanity check — v(S) değerleri:')
for name, (members, year) in test_coalitions.items():
    v_prob = compute_v(members, year, return_logit=False)
    v_logit = compute_v(members, year, return_logit=True)
    print(f'  {name}: v(prob)={v_prob:.4f}, v(logit)={v_logit:+.4f}')

## 1. Monte Carlo Shapley Analysis

Her üyenin koalisyona **marjinal katkısı**, rastgele sıralamalar üzerinden ortalama olarak hesaplanır:

$$\phi_i(v) = \frac{1}{n!} \sum_{\pi} \left[ v(S_\pi^{i} \cup \{i\}) - v(S_\pi^{i}) \right]$$

**Önemli not:** Sigmoid saturation nedeniyle raw logit kullanıyoruz (sigmoid pozitif örneklerin neredeyse hepsi için ≈1 verir; logit ayırt edici).

In [ ]:
def monte_carlo_shapley(members_cow, year, n_samples=2000, use_logit=True, seed=42):
    """Monte Carlo Shapley value tahmini.
    
    Args:
        members_cow: koalisyon üye COW kodları
        year: yıl
        n_samples: rastgele permutasyon sayısı (yüksek = doğru ama yavaş)
        use_logit: True = raw logit (önerilen), False = sigmoid prob
        seed: tekrarlanabilirlik için
    
    Returns:
        dict: {cow_code: shapley_value}
    """
    snap = snapshots[year]
    valid_members = [m for m in members_cow if m in snap['code_to_idx']]
    n = len(valid_members)
    if n < 2:
        return None
    
    rng = np.random.default_rng(seed)
    shapley = np.zeros(n)
    
    # Boş koalisyon değeri (referans)
    v_empty = 0.0  # |S|<2 için v tanımlı değil; sıfır referans alıyoruz
    
    for _ in tqdm(range(n_samples), desc=f'Shapley {year}', leave=False):
        perm = rng.permutation(n)
        prev_v = v_empty
        current_subset = []
        for step, idx in enumerate(perm):
            current_subset.append(valid_members[idx])
            if len(current_subset) < 2:
                # |S|=1: marjinal katkı = 0 (model |S|<2 için tanımlı değil)
                shapley[idx] += 0
                continue
            v_curr = compute_v(current_subset, year, return_logit=use_logit)
            if v_curr is None:
                v_curr = prev_v  # değişmedi varsay
            shapley[idx] += v_curr - prev_v
            prev_v = v_curr
    
    shapley /= n_samples
    return {valid_members[i]: shapley[i] for i in range(n)}


def print_shapley_table(shapley_dict, title, top_k=None):
    print(f'\n=== {title} ===')
    sorted_items = sorted(shapley_dict.items(), key=lambda x: -x[1])
    if top_k:
        sorted_items = sorted_items[:top_k]
    print(f'{"Rank":<5}{"COW":<6}{"Ülke":<22}{"Shapley":>10}')
    print('-' * 50)
    for i, (cow, val) in enumerate(sorted_items, 1):
        name = cow_to_name.get(cow, str(cow))
        print(f'{i:<5}{cow:<6}{name:<22}{val:+10.4f}')
    
    return sorted_items

### 1.1 Warsaw Pact — Arnavutluk'un negatif katkısı?

**Tarihsel hipotez:** Arnavutluk 1968 sonrası Varşova Paktı'ndan fiilen ayrıldı. Eski model Shapley'sinde negatif katkı vermişti. 07 modeliyle bu bulgu tekrar ediyor mu?

In [ ]:
warsaw_members = [265, 290, 310, 315, 339, 355, 360, 365]
# GDR(265), POL(290), HUN(310), CZE(315), ALB(339), BUL(355), ROM(360), USSR(365)

# Birden fazla yıl için Shapley — tarihsel evolution
for year in [1960, 1968, 1980, 1989]:
    if year not in snapshots:
        continue
    sh = monte_carlo_shapley(warsaw_members, year, n_samples=1500)
    if sh is None: continue
    print_shapley_table(sh, f'Varşova Paktı Shapley ({year}) — raw logit')

### 1.2 NATO-26 — Doğal ikili çekirdekler

**Tarihsel hipotez:** UKG-NTH (İngiltere-Hollanda) NATO içinde maksimum çekirdek bir çift. Yüksek Shapley + güçlü pair score beklenir.

In [ ]:
nato_26 = [2,20,200,210,211,212,220,230,235,255,290,310,316,317,325,
           349,350,355,360,366,367,368,385,390,395,640]

sh_nato = monte_carlo_shapley(nato_26, 2010, n_samples=1500)
print_shapley_table(sh_nato, 'NATO-26 Shapley (2010) — raw logit')

# En yüksek 5 ile en düşük 5
if sh_nato:
    sorted_sh = sorted(sh_nato.items(), key=lambda x: -x[1])
    print(f'\nEn yüksek 5:')
    for cow, val in sorted_sh[:5]:
        print(f'  {cow_to_name.get(cow, cow):<22}: {val:+.4f}')
    print(f'\nEn düşük 5:')
    for cow, val in sorted_sh[-5:]:
        print(f'  {cow_to_name.get(cow, cow):<22}: {val:+.4f}')

### 1.3 BRICS — Üye katkı sıralaması

In [ ]:
brics = [140, 365, 560, 710, 750]

for year in [2010, 2014, 2016]:
    sh = monte_carlo_shapley(brics, year, n_samples=2000)
    print_shapley_table(sh, f'BRICS Shapley ({year}) — raw logit')

### 1.4 EU-27 ve ASEAN-10

In [ ]:
eu_27 = [200,205,210,211,212,220,230,235,255,290,305,310,316,317,325,
         338,349,350,352,355,360,366,367,368,375,380,390]

asean_10 = [775,800,811,812,816,820,830,835,840,850]

sh_eu = monte_carlo_shapley(eu_27, 2010, n_samples=1500)
print_shapley_table(sh_eu, 'EU-27 Shapley (2010) — raw logit', top_k=10)

sh_asean = monte_carlo_shapley(asean_10, 2010, n_samples=2000)
print_shapley_table(sh_asean, 'ASEAN-10 Shapley (2010) — raw logit')

## 2. Blocking Coalition Analysis

**Non-superadditivity hipotezi:** Bazı $S \subsetneq N$ alt-koalisyonları için $v(S) > v(N)$ olabilir. Bu, klasik koalisyon teorisi açısından beklenmedik, ama uluslararası ilişkilerde anlamlı: koalisyon büyüdükçe iç tutarsızlık nedeniyle değer düşebilir.

Bir koalisyon $N$ ve onun değeri $v(N)$ verildiğinde:
- **Blocking coalition:** $S \subsetneq N$ ile $v(S) > v(N)$
- **% blocking:** Test edilen alt-koalisyonların kaç tanesinin blocking olduğu

Yüksek %blocking → koalisyon kırılgan, küçültülmesi değer arttırır.

In [ ]:
from itertools import combinations

def blocking_analysis(members_cow, year, max_subset_size=None, n_random_subsets=None,
                      use_logit=True, verbose=True):
    """Blocking coalition analizi.
    
    Args:
        members_cow: koalisyon üyeleri
        year: yıl
        max_subset_size: tam tarama için maks alt-küme boyutu (None=tüm boyutlar)
        n_random_subsets: yerine rastgele örnekleme (büyük N için)
        use_logit: raw logit kullan (önerilen)
        verbose: print çıktıları
    """
    snap = snapshots[year]
    valid = [m for m in members_cow if m in snap['code_to_idx']]
    n = len(valid)
    if n < 3:
        return None
    
    v_N = compute_v(valid, year, return_logit=use_logit)
    if v_N is None:
        return None
    
    # Alt-koalisyonları üret
    if n_random_subsets and 2**n > 1000:
        # Büyük N: rastgele örnekleme
        rng = np.random.default_rng(42)
        subsets = set()
        while len(subsets) < n_random_subsets:
            sz = rng.integers(2, n)
            sub = tuple(sorted(rng.choice(valid, sz, replace=False)))
            subsets.add(sub)
        subsets = [list(s) for s in subsets]
    else:
        # Tam tarama (küçük N)
        max_sz = min(max_subset_size or n-1, n-1)
        subsets = []
        for k in range(2, max_sz + 1):
            for combo in combinations(valid, k):
                subsets.append(list(combo))
    
    blocking = []
    all_results = []
    for sub in subsets:
        v_S = compute_v(sub, year, return_logit=use_logit)
        if v_S is None: continue
        all_results.append((sub, v_S))
        if v_S > v_N:
            blocking.append((sub, v_S, v_S - v_N))
    
    if verbose:
        print(f'  Grand coalition v(N) = {v_N:+.4f} (n={n})')
        print(f'  Test edilen alt-koalisyon: {len(all_results)}')
        print(f'  Blocking (v(S)>v(N)): {len(blocking)} ({100*len(blocking)/max(len(all_results),1):.1f}%)')
        if blocking:
            top_blocks = sorted(blocking, key=lambda x: -x[2])[:5]
            print(f'\n  En güçlü 5 blocking koalisyon:')
            for sub, v_S, diff in top_blocks:
                sub_names = ', '.join(cow_to_name.get(m, str(m)) for m in sub[:6])
                if len(sub) > 6: sub_names += f' (+{len(sub)-6})'
                print(f'    |S|={len(sub):2d}, v(S)={v_S:+.4f}, Δ={diff:+.4f}: {sub_names}')
    
    return {
        'v_N': v_N, 'n': n,
        'n_tested': len(all_results),
        'n_blocking': len(blocking),
        'pct_blocking': 100 * len(blocking) / max(len(all_results), 1),
        'top_blocking': sorted(blocking, key=lambda x: -x[2])[:10],
    }

In [ ]:
# Tüm büyük koalisyonlar için blocking analizi
coalitions_to_test = [
    ('Warsaw Pact (1980)',  warsaw_members, 1980),
    ('Warsaw Pact (1989)',  warsaw_members, 1989),
    ('BRICS (2014)',        brics,           2014),
    ('ASEAN-10 (2010)',     asean_10,        2010),
    ('NATO-26 (2010)',      nato_26,         2010),
    ('EU-27 (2010)',        eu_27,           2010),
]

blocking_results = {}
for name, members, year in coalitions_to_test:
    print(f'\n=== Blocking Analysis: {name} ===')
    # Büyük koalisyonlar için random subset; küçükler için tam tarama
    if len(members) > 12:
        result = blocking_analysis(members, year, n_random_subsets=500, use_logit=True)
    else:
        result = blocking_analysis(members, year, max_subset_size=len(members)-1, use_logit=True)
    blocking_results[name] = result

## 3. Özet Tablosu (paylaşılabilir)

In [ ]:
print('='*72)
print('GAME-THEORETIC ANALYSIS — ÖZET (07 modeli üzerinden)')
print('='*72)

print('\n## Blocking Coalition Analysis')
print(f'\n| Coalition          |  n | v(N) raw  | n_tested | n_block | % block |')
print(f'|--------------------|----|-----------|----------|---------|---------|')
for name, res in blocking_results.items():
    if res is None: continue
    print(f'| {name:<18} | {res["n"]:2d} | {res["v_N"]:+9.4f} | {res["n_tested"]:8d} | {res["n_blocking"]:7d} | {res["pct_blocking"]:6.1f}% |')

print('\n## Şimdi yorumlanmalı:')
print('  1. Warsaw Pact 1980 vs 1989: blocking yüzdesi yükseliyor mu? (çöküş öncesi kırılganlık)')
print('  2. BRICS vs NATO: hangi koalisyon daha kırılgan (daha çok blocking)?')
print('  3. Shapley sonuçları: Arnavutluk Warsaw\'da hâlâ en düşük mü?')
print('  4. NATO içinde maksimum çekirdek hangi çift?')

## 📋 Paylaşılabilir Metin Özeti

Bu hücreyi çalıştır, çıktıyı Claude'a kopyala-yapıştır.

In [ ]:
# === 10_GAME_THEORY — PAYLAŞIM ÖZETİ ===
print('='*72)
print('GAME-THEORETIC ANALYSIS ÖZETİ — Claude ile paylaşmak için kopyala')
print('='*72)

# Sanity check (model çalışıyor mu)
print('\n### Sanity check — v(S) ###')
for name, (members, year) in test_coalitions.items():
    v_p = compute_v(members, year, return_logit=False)
    v_l = compute_v(members, year, return_logit=True)
    print(f'  {name:30s}: prob={v_p:.4f}, logit={v_l:+.4f}')

# Shapley özetleri
shapley_results = {
    'Warsaw Pact 1980': (warsaw_members, 1980),
    'Warsaw Pact 1989': (warsaw_members, 1989),
    'NATO-26 2010': (nato_26, 2010),
    'EU-27 2010': (eu_27, 2010),
    'ASEAN-10 2010': (asean_10, 2010),
    'BRICS 2014': (brics, 2014),
}

for name, (members, year) in shapley_results.items():
    sh = monte_carlo_shapley(members, year, n_samples=1000)
    if sh is None: continue
    sorted_sh = sorted(sh.items(), key=lambda x: -x[1])
    print(f'\n### Shapley — {name} (logit) ###')
    print('| Rank | Country               |  Shapley  |')
    print('|------|------------------------|-----------|')
    # Top 3 + Bottom 3
    for i, (cow, val) in enumerate(sorted_sh[:3], 1):
        nm = cow_to_name.get(cow, str(cow))[:22]
        print(f'| {i:4d} | {nm:22s} | {val:+9.4f} |')
    if len(sorted_sh) > 6:
        print('|  ... | ...                    |       ... |')
    for i, (cow, val) in enumerate(sorted_sh[-3:], len(sorted_sh)-2):
        nm = cow_to_name.get(cow, str(cow))[:22]
        print(f'| {i:4d} | {nm:22s} | {val:+9.4f} |')

# Blocking özetleri
print('\n### Blocking Analysis Tablosu ###')
print('| Coalition          |  n | v(N) logit | n_tested | n_block | % block |')
print('|--------------------|----|------------|----------|---------|---------|')
for name, res in blocking_results.items():
    if res is None: continue
    print(f'| {name:<18} | {res["n"]:2d} | {res["v_N"]:+10.4f} | {res["n_tested"]:8d} | {res["n_blocking"]:7d} | {res["pct_blocking"]:6.1f}% |')

print('\n### En güçlü blocking koalisyonlar (her ana koalisyon için top 3) ###')
for name, res in blocking_results.items():
    if res is None or not res['top_blocking']: continue
    print(f'\n{name}:')
    for i, (sub, v_S, diff) in enumerate(res['top_blocking'][:3], 1):
        sub_names = ', '.join(cow_to_name.get(m, str(m)) for m in sub[:5])
        if len(sub) > 5: sub_names += f' (+{len(sub)-5})'
        print(f'  {i}. |S|={len(sub)}, v(S)={v_S:+.4f}, Δ={diff:+.4f}: {sub_names}')

print('\n' + '='*72)

## 4. Nash Bargaining Solution & Nucleolus

Shapley'e ek iki kooperatif çözüm kavramı:

- **NBS (Nash Bargaining Solution)** — pazarlık-tabanlı fair division. İki versiyon:
  - *Egalitarian*: $d_i=0$ → eşit dağılım (trivial referans)
  - *Marginal-threat*: $d_i = \min_S (v(S) - v(S\setminus\{i\}))$ → her oyuncunun min marjinal katkısı threat
  - Kapalı form: $x_i^* = d_i + (v(N) - \sum_j d_j)/n$

- **Nucleolus (least-core)** — stabilite-tabanlı. LP ile $\min \epsilon$ s.t. $\sum_{i\in S} x_i \geq v(S) - \epsilon$ ∀S.
  - $\epsilon \leq 0$ → klasik core var (stabil)
  - $\epsilon > 0$ → core boş, non-superadditive (beklenen)

In [ ]:
def nbs_egalitarian(members, year, use_logit=True):
    """Egalitarian NBS: d_i=0 → eşit dağılım."""
    v_N = compute_v(members, year, return_logit=use_logit)
    if v_N is None: return None
    return {m: v_N / len(members) for m in members}


def compute_min_marginal_contribution(members, year, use_logit=True, n_samples=100, seed=42):
    """Her üye için S ⊃ {i} arasında min marjinal katkı."""
    rng = np.random.default_rng(seed)
    threats = {}
    n = len(members)
    for m in members:
        other = [x for x in members if x != m]
        min_marg = float('inf')
        for _ in range(n_samples):
            k = rng.integers(1, n)
            sub = list(rng.choice(other, min(k, len(other)), replace=False))
            S_with = sub + [m]
            S_without = sub
            v_with = compute_v(S_with, year, return_logit=use_logit)
            v_without = compute_v(S_without, year, return_logit=use_logit) if len(S_without) >= 2 else 0.0
            if v_with is not None and v_without is not None:
                marg = v_with - v_without
                if marg < min_marg:
                    min_marg = marg
        threats[m] = min_marg if min_marg != float('inf') else 0.0
    return threats


def nbs_with_threats(members, year, use_logit=True, n_samples=100):
    """NBS: x_i = d_i + (v(N) - Σd) / n"""
    v_N = compute_v(members, year, return_logit=use_logit)
    if v_N is None: return None, None
    threats = compute_min_marginal_contribution(members, year, use_logit, n_samples)
    total_threat = sum(threats.values())
    surplus = v_N - total_threat
    n = len(members)
    allocation = {m: threats[m] + surplus / n for m in members}
    return allocation, threats


print('NBS fonksiyonları tanımlandı')


In [ ]:
from itertools import combinations
from scipy.optimize import linprog


def least_core(members, year, use_logit=True, max_coalitions=2000, seed=42):
    """Least-core LP (nucleolus 1-step). Küçük n için tam, büyük için örneklem."""
    n = len(members)
    v_N = compute_v(members, year, return_logit=use_logit)
    if v_N is None: return None

    coalition_specs = []

    if 2**n <= max_coalitions + n + 2:
        for k in range(2, n):
            for combo in combinations(range(n), k):
                sub_members = [members[i] for i in combo]
                v_S = compute_v(sub_members, year, return_logit=use_logit)
                if v_S is not None:
                    indicator = [1.0 if i in combo else 0.0 for i in range(n)]
                    coalition_specs.append((indicator, v_S))
        method_used = 'exact'
    else:
        rng = np.random.default_rng(seed)
        seen = set()
        attempts = 0
        while len(coalition_specs) < max_coalitions and attempts < max_coalitions * 3:
            attempts += 1
            k = int(rng.integers(2, n))
            combo = tuple(sorted(rng.choice(n, k, replace=False).tolist()))
            if combo in seen: continue
            seen.add(combo)
            sub_members = [members[i] for i in combo]
            v_S = compute_v(sub_members, year, return_logit=use_logit)
            if v_S is not None:
                indicator = [1.0 if i in combo else 0.0 for i in range(n)]
                coalition_specs.append((indicator, v_S))
        method_used = f'sampled ({len(coalition_specs)})'

    if not coalition_specs:
        return None

    # min epsilon s.t. Σ_{i∈S} x_i + epsilon ≥ v(S)
    c = [0.0]*n + [1.0]
    A_ub = []
    b_ub = []
    for indicator, v_S in coalition_specs:
        A_ub.append([-x for x in indicator] + [-1.0])
        b_ub.append(-v_S)
    A_eq = [[1.0]*n + [0.0]]
    b_eq = [v_N]
    bounds = [(None, None)] * (n + 1)

    res = linprog(c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

    if not res.success:
        print(f'  LP failed: {res.message}')
        return None

    return {
        'allocation': {members[i]: res.x[i] for i in range(n)},
        'epsilon': res.x[-1],
        'core_exists': res.x[-1] <= 1e-6,
        'method': method_used,
        'n_coalitions': len(coalition_specs),
        'v_N': v_N,
    }


print('Least-core / nucleolus fonksiyonu tanımlandı')


In [ ]:
# Tüm koalisyonlar için NBS + Nucleolus
allocation_results = {}

scenarios = [
    ('BRICS 2014',       brics,    2014),
    ('Warsaw Pact 1980', warsaw_members, 1980),
    ('Warsaw Pact 1989', warsaw_members, 1989),
    ('ASEAN-10 2010',    asean_10, 2010),
    ('NATO-26 2010',     nato_26,  2010),
    ('EU-27 2010',       eu_27,    2010),
]

for title, members, year in scenarios:
    print(f'\n>>> {title}...')
    nbs_eg = nbs_egalitarian(members, year)
    nbs_th_alloc, threats = nbs_with_threats(members, year, n_samples=80)
    nuc = least_core(members, year, max_coalitions=800)

    allocation_results[title] = {
        'members': members, 'year': year,
        'nbs_eg': nbs_eg, 'nbs_th': nbs_th_alloc,
        'threats': threats, 'nucleolus': nuc,
    }

    if nuc:
        print(f'  Least-core epsilon = {nuc["epsilon"]:+.4f}, '
              f'core_exists = {nuc["core_exists"]}, method = {nuc["method"]}')


In [ ]:
# Allocation tablolarını yazdır (Shapley 10'dan, NBS + Nucleolus burada)
def print_allocation_comparison(title, members, year, allocs):
    """Shapley + NBS + Nucleolus karşılaştırması."""
    v_N = compute_v(members, year, return_logit=True)
    print(f'\n=== {title} (v(N) logit = {v_N:+.4f}) ===')

    # Shapley'i tekrar hesapla (kısa, 800 sample)
    sh = monte_carlo_shapley(members, year, n_samples=800)

    nbs_th = allocs['nbs_th']
    nuc = allocs['nucleolus']
    nbs_eg_v = allocs['nbs_eg'][members[0]]  # hepsi eşit

    print(f'  Egalitarian NBS (hepsi eşit): {nbs_eg_v:+.4f} per üye')
    if nuc:
        print(f'  Least-core epsilon: {nuc["epsilon"]:+.4f}  ({"core var" if nuc["core_exists"] else "core boş, non-superadditive"})')

    print(f'\n  {"Üye":<22} {"Shapley":>10} {"NBS_th":>10} {"threat":>10} {"Nuc":>10}')
    print('  ' + '-' * 70)

    rows = []
    for m in members:
        name = cow_to_name.get(m, str(m))[:20]
        s = sh.get(m, 0) if sh else 0
        nt = nbs_th[m] if nbs_th else 0
        th = allocs['threats'].get(m, 0) if allocs['threats'] else 0
        nc = nuc['allocation'].get(m, 0) if nuc else 0
        rows.append((name, s, nt, th, nc))

    rows.sort(key=lambda r: -r[1])  # Shapley'e göre

    for name, s, nt, th, nc in rows:
        print(f'  {name:<22} {s:+10.4f} {nt:+10.4f} {th:+10.4f} {nc:+10.4f}')

    # Sanity: toplamlar v(N)'e eşit olmalı
    sum_sh = sum(r[1] for r in rows)
    sum_nt = sum(r[2] for r in rows)
    sum_nc = sum(r[4] for r in rows)
    print(f'  {"TOPLAM":<22} {sum_sh:+10.4f} {sum_nt:+10.4f} {" ":>10} {sum_nc:+10.4f}')
    print(f'  (efficiency: hepsi v(N)={v_N:+.4f} olmalı)')

for title, allocs in allocation_results.items():
    print_allocation_comparison(title, allocs['members'], allocs['year'], allocs)


In [ ]:
# Çözüm kavramları arası korelasyon
print('\n=== Çözüm Kavramları Arası Korelasyon (Pearson) ===')
print(f'{"Koalisyon":<22} {"Shap-NBS":>10} {"Shap-Nuc":>10} {"NBS-Nuc":>10}')
print('-' * 57)

for title, allocs in allocation_results.items():
    members = allocs['members']
    year = allocs['year']
    sh = monte_carlo_shapley(members, year, n_samples=600)
    if sh is None: continue

    sh_vec = np.array([sh.get(m, 0) for m in members])
    nbs_vec = np.array([allocs['nbs_th'][m] for m in members])

    c_sh_nbs = np.corrcoef(sh_vec, nbs_vec)[0, 1]
    if allocs['nucleolus']:
        nuc_vec = np.array([allocs['nucleolus']['allocation'].get(m, 0) for m in members])
        c_sh_nuc = np.corrcoef(sh_vec, nuc_vec)[0, 1]
        c_nbs_nuc = np.corrcoef(nbs_vec, nuc_vec)[0, 1]
    else:
        c_sh_nuc = c_nbs_nuc = float('nan')

    print(f'{title:<22} {c_sh_nbs:>10.3f} {c_sh_nuc:>10.3f} {c_nbs_nuc:>10.3f}')


## 📋 NBS + Nucleolus Paylaşım Özeti

In [ ]:
# === NBS + NUCLEOLUS PAYLAŞIM ÖZETİ ===
print('=' * 78)
print('NBS + NUCLEOLUS — Claude ile paylaşmak için kopyala-yapıştır')
print('=' * 78)

print('\n## Least-core ε Tablosu')
print('| Koalisyon          | n  | v(N) logit | ε        | core var? | method     |')
print('|--------------------|----|------------|----------|-----------|------------|')
for title, allocs in allocation_results.items():
    nuc = allocs['nucleolus']
    if nuc is None: continue
    n = len(allocs['members'])
    core_str = 'EVET' if nuc['core_exists'] else 'HAYIR'
    print(f'| {title:<18} | {n:2d} | {nuc["v_N"]:+10.4f} | {nuc["epsilon"]:+8.4f} | {core_str:9s} | {nuc["method"]:10s} |')

print('\n## Allocation Tabloları (Shapley + NBS + Nucleolus)')
for title, allocs in allocation_results.items():
    members = allocs['members']
    year = allocs['year']
    sh = monte_carlo_shapley(members, year, n_samples=600)
    if sh is None: continue

    rows = []
    for m in members:
        name = cow_to_name.get(m, str(m))[:18]
        s = sh.get(m, 0)
        nt = allocs['nbs_th'][m]
        nc = allocs['nucleolus']['allocation'].get(m, 0) if allocs['nucleolus'] else None
        rows.append((name, s, nt, nc))
    rows.sort(key=lambda r: -r[1])

    print(f'\n### {title}')
    print('| Üye                | Shapley   | NBS_threat | Nucleolus  |')
    print('|--------------------|-----------|------------|------------|')

    if len(rows) <= 10:
        show = rows
    else:
        show = rows[:3] + [(' ... ', None, None, None)] + rows[-3:]

    for r in show:
        if r[1] is None:
            print(f'| {r[0]:<18} |  ...      |  ...       |  ...       |')
        else:
            nuc_str = f'{r[3]:+10.4f}' if r[3] is not None else '       N/A'
            print(f'| {r[0]:<18} | {r[1]:+9.4f} | {r[2]:+10.4f} | {nuc_str} |')

print('\n' + '=' * 78)
